# 11. Fault-tolerant Workflows

## 11.01. `POST /admin/newsletters` - A refreshser

##### 11.01.0.0. Skimming: What did you notice and why? Any Questions

**What?**  
- We we want a **best effort newsletter delivery**: where the newsletter is delivered to all confirmed subscribers with minimal duplicate deliveries
- Recap of our `publish_newsletter` handler that is responsible for our newsletter delivery.

**Why?**  
- Our application as of now is susceptible to transient failures like
    - Application crashes
    - PostMark errors
    - network failures 
- Because we
    1. Fetch all subscribers from the database
    2. Loop over all the subscribers and send the newsletter issue one by one
    
  What happens when one of the email sends causes an application crash or PostMark error? We end up in a situation where the newsletter is delivered to  
  some subscribers and not to others. We also have no idea who received and who didn't and why. Additionally we have no mechanism in place to retry delivery
  of only the failures.


**Questions?**  
Why can we not rule out duplicate delivery entirely?  
> Look out for where this is discussed


##### 11.01.0.1. Deep Dive: Summarize, ELI5, Connect

## 11.2. Our Goal

A **best-effort delivery**.

## 11.3. Failure Modes

##### 11.03.0.0. Skimming: What did you notice and why? Any Questions

**What?**  
- 4 main types of failure modes
    1. Invalid Inputs - We are using the `Form` extractor and redirect back to the `publish_newsletter_form`. Currently not bad. In the future
       we might have more advanced validation requirements
    2. Network I/O
       - Postgres - Did postgress fail before we had fetched all the confirmed subscribers or all.
       - Postmark API  - Did our postmark email delivery fail at the first confirmed subscriber or at the $m-n+1$ subscriber.
         > Where $m \rightarrow$ total number of subscribers  
         > & $n \rightarrow$ subscribers to whom newsletters where delivered successfully.
    3. Application Crashes - Since we are hosting our application in the cloud, there may be an issue with our cloud provider servers leading to issues
       with our application. When sending the newsletter we have no visibility currently at what point our server crashed. It might have been at n-1 confirmed
       subscriber email send.
    4. Author actions - We don't have a mechanism of showing current newsletter delivery progress, so an author might click publish multiple times if they are
       not getting feedback from the application. I.e. our application is currently not retry safe.


**Why?**  
Really important considerations for making our `publish_newsletter` endpoint retry safe and **idempotent**.


**Questions?**  
None

##### 11.03.0.1. Deep Dive: Summarize, ELI5, Connect

### 11.3.1. Invalid Inputs.

### 11.3.2. Network I/O

### 11.3.3. Application Crashes

### 11.3.4. Author Actions

## 11.4. Idempotency: An Introduction

##### 11.04.0.0. Skimming: What did you notice and why? Any Questions

**What?**  
- Idempotency definition
  > **Mine**  
  > The property of an operation or a function call resulting in to the same initial outcome/result regardless of multiple repeated calls  
  > e.g. $1\times1$ is idempotent. $1 + 1$ is not.
  >
  > **Google**  
  > The property of certain operations or functions where applying them multiple times has the same exact effect as applying them just once.
  >
  > **Book**  
  > An API endpoint is idempotent(or _**retry-safe**_) if the caller has no way to **observe** if a request
  > has been sent to the server once or multiple times.

- How can we distinguish between a user retry and a distinct separate call? The caller generates an idempotency key that helps the server more easily distinguish the caller's intent.

- Concurrent requests? $\rightarrow$ we introduce **synchronization** the secode request should not be processed until the first one has completed.

**Why?**  
I've seen the terminology multiple times and had a sense of what it meant now I can be sure.

**Question**  
None

##### 11.04.0.1. Deep Dive: Summarize, ELI5, Connect

### 11.4.0. Overview.

### 11.4.1. Idempotency In Action: Payments

### 11.4.2. Idempotency Keys

### 11.4.3. Concurrent Request

## 11.5. Requirements As Tests #1

##### 11.05.0.0. Skimming: What did you notice and why? Any Questions

**What?**  
- `app.test_user.login()` in the `newsletter_creation_is_idempotent` test.
- `expected(1)` in the the `Mock`
- 2 `post_publish_newsletter` requests with the same `newsletter_request_body`


**Why?**  
- Had been thinking about creating a helper method around authenticated route testing. Also it interesting how `TestApp` has a dependency on `TestUser`
  and `TestUser`'s  `login` method has a dependency on `TestApp`. I guess because we are using componsition instead of inheritance we don't get any issues.
- We expect that our `Mock` email server will only receive a request once if our implememntation is idempotent.
- The 2nd `post_publish_newsletter` request is how we test a retry and **retry-safety**.

**Questions?**  



##### 11.05.0.1. Deep Dive: Summarize, ELI5, Connect

### 11.5.0. Overview

## 11.6. Implementation Strategies

##### 11.06.0.0. Skimming: What did you notice and why? Any Questions

**What?**  
- Stateful vs Stateless approach when it comes to implementing retry safety.
- How the 2 approaches handle change in state before a retry request.
- We go with the stateful approach because Postmark doesn't not have any idempotent retry mechanism.


**Why?**  
- A stateful strategy (Save and Replay) means we implement persistence for our idempotency mechanism in that every retry request is checked against an internal
  store that tracks against the stored state of the initial request.
- A stateless strategy (Deterministic Key Generation) means we rely on the external service's externam mechanism only provided an idemptency key as part of the request so that the external
  service can distinguish the original request from a retry.
- A stateful approach checks a retry against the request state i.e. is this request similar to the previous one.  
- A stateless approach is checked against an application state. i.e if the application state changed before a retry, the request to the external service includes this state in the retry request.
  > The example used here is for example if a new confirmed subscriber joins the newsletter. In a stateless approach if a retry is triggered, the request would include the new subscriber email  
  > send.
  > In the stateful approach we are just checking if the retry idempotency key is already stored and we return the previosly stored response, withoug having to check the application state.

**Question?**  
- I'm curious if its either strictly either, or. Are the situations where is we need both stateful and stateless implementation?




##### 11.06.0.1. Deep Dive: Summarize, ELI5, Connect

### 11.6.1. Stateful Idempotency: Save and Replay

### 11.6.2. Stateless Idempotency: Deterministic Key Generation

### 11.6.3. Time Is A Tricky Beast

### 11.6.4. Making a Choice

## 11.7. Idempotency Store

##### 11.07.0.0. Skimming: What did you notice and why? Any Questions

**What?**
- The decision to use Postgres as our idempotency store.
- Composite type for headers but no composite type for the `httt_response` as a whole.
- `header_pair` is `CREATE TYPE`. 

**Why?**
- The author spares us from exploring the Redis route only to find the limitations that would lead us to work with postgres in the first place as our idempotency store. Looking forwared to unpacking this.
- Working with Postgres composite types looks interesting.
- Had missed `CREATE TYPE` SQL command


**Questions?**  

- The book has the following excerpt.
> _"We could have defined an overall http_response composite type, but we would have run into a bug in sqlx
which is in turn caused by a bug in the Rust compiler."_

    Curious if the bug is fixed. Amazing that this is possible due to ai. Heres what I got from [claude](https://claude.ai/share/8e0c3534-fe7d-4ee3-98e7-b18808a041cc)

- Does the idempotency composite primary key order matter? `PRIMARY KEY(user_id, idempotency_key)` vs `PRIMARY KEY( idempotency_key, user_id)`.  
Heres a [google](https://share.google/aimode/1ZJlJIxx0ji4Hr4F1) exploration of the question.


##### 11.07.0.1. Deep Dive: Summarize, ELI5, Connect

### 11.7.1. Which Database Should We Use?

### 11.7.2. Schema

## 11.8. Save and Replay (Slightly Bulky)

##### 11.08.0.1. Deep Dive: Summarize, ELI5, Connect

### 11.8.1. Read Idempotency Key

##### 11.08.0.0. Skimming: What did you notice and why? Any Questions

**What?**  
- Was curious how we set the `idempotency_key`
- Beware of the test that will fail because of new `idempotency_key` field in `FormData`
- `e400` util helper for bad request.
- `TryFrom`, `From` and `AsRef` implementations for `IdempotencyKey`


**Why?**  
- We just use a regular `UUID::new_v4().to_string()`. In our case we will use `UUID::now_v7().to_string()` prefering a `PRIMARY KEY(idempotency_key, user_id)` ordering
  for faster searches and ordering on retries rather than on `user_id` requests.
- As in the previous chapter where we used either `AppError::Unexpected` to substitute for the opaque `e500` in our axum implementation, thinking we'll default to `AppError::BadRequest`
  for the opaque `e400` in this chapter.
- `TryFrom` for converting a string to our custom type for validation. `From` for converting our `IdemptencyKey` type to a plain `String` during persistence. `AsRef` for share semantics.
  > Need to validate if this is the correct mental model.


**Questions?**  
None


##### 11.08.0.1. Deep Dive: Summarize, ELI5, Connect

### 11.8.2. Retrieve Saved Responses

##### 11.08.0.0. Skimming: What did you notice and why? Any Questions

**What?**  
- `#[derive(Debug, sqlx::Type)]` & `#[sqlx(type_name="header_pair")]` macro
- `respose_headers as "response_headers: Vec<HeaderPairRecord>"`

**Why?**  
- The first 2 macros allows sqlx to know how to handle the postgres **composite type** as part of the retreive query
- We have to use explicit type annotation for `sqlx::query` to be able to handle the custom composite type.


**Questions?**  



##### 11.08.0.1. Deep Dive: Summarize, ELI5, Connect

### 11.8.3. Save Responses

##### 11.08.0.0. Skimming: What did you notice and why? Any Questions

**What?**  
- `.body()` is generic over `B` which defaults to `BoxBody` if no type is specified.
- `MessageBody` trait that must be implemented for any type that is intended to be used as a body in `actix-web`
- Because of streaming alternative we need an owned instance of `HttpResponse` which implements `MessageBody` that we breakdown using `into_parts`. We can then use
  `actix_web::body::into_bytes` to get type body type that we can store. We then have to reassemble the http response back using `set_body` on the headers to be able
  to return a HttpResponse
- `query_unchecked!` macro and `PgHasArrayType` trait for `HeaderPairRecord`

**Why?**  
- HTTP/1.1 specifies that a body can be fully-formed or streamed. `BoxBody` is an enum representing this strategy.
- How we dive into `response_body` details inorder to understand how best to parse it for storage then reassamble the it back for our user/client side HttpResponse
- We use `query_unchecked!` becuase `query!` is not powerful enough to check the query against our custom composite SQL type.
- We implement `PgHasArrayType` for `HeaderPairRecord` to provide sqlx with type information on our `header_pair` column.

**Questions?**  
- Was curious if `query!` is now powerful enough to use without having to disable compile-time verifcation using `query_unchecked`.  
Here's what I from [claude](https://claude.ai/share/8e0c3534-fe7d-4ee3-98e7-b18808a041cc).
> It truly amazing how much we are able to learn, discover, experiment with AI.


#### 11.8.3.0. Intro

#### 11.8.3.1. `MessageBody` and HTTP Streaming

#### 11.8.3.2. Array Of Composite Postgres Types

#### 11.8.3.3. Plug it in

## 11.9. Concurrent Request

##### 11.09.0.0. Skimming: What did you notice and why? Any Questions

**What?**  
- **cross-request synchronization**
- We relax the `NOT NULL` constraints for our `response_` fields for us to be able to insert into the `indepotency` table before a response is returned.
- Postgres `ON CONFLICT` statement
- `as "response_[placeholder]!"`
- Different transaction isolation levels. We'll need to look into this.


**Why?**  
- What mechanism will we use to facilitate **cross-request synchronization**? Turns out we'll use _**transaction isolation levels**_ by sharing a transaction
  between our `INSERT` statement in `try_processing` and our `UPDATE` statement in `save_response`, ensuring our first request is only read after being fully processed, such that
  subsequent request either return what is already committed or `DO NOTHING` until the first transactions is complete.


**Questions?**
- Wondering if instead of `ON CONFLICT DO NOTHING` it would have been better to have a `indempotency_status` field that has `PENDING` and `SET`. I guess i understand
  why `DO NOTHING` is a better approach because, the first retry can update the status to `PENDING`. What about the second and the third. Leading to unnecessary writes
  or unnecessary branching logic.


##### 11.09.0.1. Deep Dive: Summarize, ELI5, Connect

### 11.9.0. Overview

### 11.9.1. Requirements As Tests #2

### 11.9.2. Synchromization

## 11.10. Dealing With Errors (Slightly Bulky)

##### 11.10.0.0. Skimming: What did you notice and why? Any Questions

##### 11.10.0.1. Deep Dive: Summarize, ELI5, Connect

### 11.10.0. Overview

### 11.10.1. Distrubuted Transactions

### 11.10.2. Backward Recovery

### 11.10.3. Forward Recovery

### 11.10.0. Asynchronous Processing

## 11.x. Epilogue

## 11.12. Axum Implementation Notes. 